[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-logistic.ipynb)

# Logistic Regression & Classification

*AIBits Academy · Machine Learning End To End · Classification*

Despite the name, logistic regression is a classification algorithm — it models the probability that a sample belongs to a class using the sigmoid function.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

> **🎯 Intuition First**
>
> Ordinary regression outputs any number — but "this customer will subscribe" isn't a number, it's a **probability between 0 and 1**. Logistic regression keeps the familiar linear engine, then squashes its raw score through an S-shaped curve (the **sigmoid**) that bends any value, however large or small, into the 0–1 range. Read the result as "how likely"; a threshold (usually 0.5) turns that into a yes/no call.

> **📋 Real-World Case Study — Bank Marketing Campaign Response**
>
> A classic banking use case: given a customer's age, account balance, prior contact history, and campaign channel, predict the probability they'll subscribe to a term deposit after a call. Logistic regression is a natural fit here — beyond the yes/no prediction, its coefficients are directly interpretable ("each additional prior successful contact multiplies subscription odds by e^θ"), letting a marketing team understand *why* a customer is likely to respond, not just whether — exactly the explainability a purely black-box model wouldn't offer for the same task.

## Why Not Linear Regression for Classification?

Linear regression can predict values outside [0,1], violating probability constraints. It also treats the problem as metric regression — a wrong prediction of class 2 is "worse" than predicting class 1, even for a binary problem. Logistic regression fixes both by squashing output through the sigmoid function.

## The Sigmoid Function

$$\sigma(z) = \frac{1}{1+e^{-z}} \quad\text{where } z = \mathbf{x}^{\top}\theta$$

σ(z) always lies in (0,1), interpreted as P(y=1 | x, θ). We predict class 1 if P > 0.5 (default threshold).

## Loss Function — Binary Cross-Entropy

MSE is not convex for logistic regression. Instead we use log-loss (binary cross-entropy):

$$J(\theta) = -\frac{1}{m}\sum_i\big[y_i\log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i)\big]$$

This penalises confident wrong predictions harshly (log(0) → ∞). The loss is convex in θ, so gradient descent finds the global minimum.

> **📊 Prerequisite refresher**
>
> This log-loss isn't an arbitrary choice of penalty: minimising it is mathematically identical to Maximum Likelihood Estimation under a Bernoulli model, exactly as derived on the **Inferential Statistics** prerequisite page's MLE section. Fitting a logistic regression is finding the θ that makes the observed 0/1 outcomes most probable — not a separate idea from MLE, but a direct application of it.

## Deriving the Gradient — Why It Looks Just Like Linear Regression's

Differentiating J(θ) requires the chain rule through σ(z), using the elegant identity σ′(z) = σ(z)(1−σ(z)):

$$\frac{\partial J}{\partial \theta_j} = \frac{1}{m}\sum_i(\hat{y}_i-y_i)\cdot x_{ij} \quad\Longrightarrow\quad \nabla J(\theta) = \frac{1}{m}\mathbf{X}^{\top}(\hat{y}-\mathbf{y}) \quad\text{where } \hat{y}=\sigma(\mathbf{X}\theta)$$

This is *identical in form* to linear regression's gradient — only the definition of ŷ changed (linear vs. sigmoid). This isn't a coincidence: both are special cases of the **Generalized Linear Model** framework, where any distribution from the exponential family (Gaussian → linear regression, Bernoulli → logistic regression, Poisson → Poisson regression) paired with its natural link function produces this same clean gradient form. There is no closed-form solution here (unlike the Normal Equation) — the log-likelihood has no algebraic maximum, so gradient-based methods are required.

## From Scratch with NumPy

In [ ]:
# Logistic Regression from scratch — HDFC Bank loan default prediction
import numpy as np

np.random.seed(42)
n = 800
income   = np.random.normal(50, 15, n)     # ₹ thousands/month
cibil    = np.random.normal(700, 80, n)    # credit score
emi_ratio= np.random.uniform(0.1, 0.6, n)  # EMI / income
logit = (0.04*(income-50) + 0.005*(cibil-700)
         - 4*emi_ratio + np.random.normal(0,0.5,n))
y = (logit > 0).astype(float)       # 1 = no default

X = np.column_stack([np.ones(n), income, cibil, emi_ratio])
X[:, 1:3] = (X[:, 1:3] - X[:, 1:3].mean(0)) / X[:, 1:3].std(0)

def sigmoid(z): return 1/(1+np.exp(-np.clip(z,-500,500)))

theta = np.zeros(X.shape[1])
alpha, iters, m = 0.1, 1000, len(y)

for _ in range(iters):
    y_hat = sigmoid(X @ theta)
    grad  = X.T @ (y_hat - y) / m
    theta -= alpha * grad

print("Learned θ:", np.round(theta, 3))

# Evaluate
preds = (sigmoid(X @ theta) >= 0.5).astype(int)
acc = np.mean(preds == y)
tp  = np.sum((preds==1) & (y==1))
fp  = np.sum((preds==1) & (y==0))
fn  = np.sum((preds==0) & (y==1))
precision = tp/(tp+fp)
recall    = tp/(tp+fn)
f1        = 2*precision*recall/(precision+recall)
print(f"Accuracy={acc:.3f}  Precision={precision:.3f}  Recall={recall:.3f}  F1={f1:.3f}")

## With scikit-learn — Multi-class (Softmax)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
import numpy as np

# Zomato order category: 0=Budget, 1=Mid, 2=Premium
np.random.seed(7)
n = 600
age      = np.random.randint(18,55,n)
order_val= np.random.randint(80,1500,n)  # ₹
freq     = np.random.randint(1,30,n)      # orders/month
X_mc = np.column_stack([age, order_val, freq])
y_mc = (order_val // 450).clip(0,2)

X_tr, X_te, y_tr, y_te = train_test_split(X_mc, y_mc, test_size=0.2, random_state=42)
sc = StandardScaler()
model = LogisticRegression(solver='lbfgs', max_iter=500)   # lbfgs is multinomial by default
model.fit(sc.fit_transform(X_tr), y_tr)
print(classification_report(y_te, model.predict(sc.transform(X_te)),
      target_names=['Budget','Mid','Premium']))

## Key Hyperparameters

| Parameter | Options / Range | Effect |
|---|---|---|
| C (inverse of λ) | 0.001 – 1000 | Lower C = stronger regularisation |
| penalty | l1, l2, elasticnet | l1 → sparsity; l2 → stability |
| solver | lbfgs, liblinear, saga | saga needed for l1 + large data |
| class_weight | balanced / dict | Handle class imbalance |
| threshold | 0.1 – 0.9 | Trade precision vs recall (not a sklearn param — applied post-predict_proba) |

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · The sigmoid

Write `sigmoid(z)` = 1 / (1 + e^-z) with NumPy so it works on arrays.

In [ ]:
import numpy as np
def sigmoid(z):
    pass   # TODO


In [ ]:
try:
    check("sigmoid(0) = 0.5", sigmoid(0) == 0.5)
    check("large positive -> 1", sigmoid(20) > 0.999999)
    check("works on arrays", np.allclose(sigmoid(np.array([-1.0, 1.0])), [0.268941, 0.731059], atol=1e-5))
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

```

</details>

### Exercise 2 · Medium · Fit and predict probabilities

Fit a `LogisticRegression` on `hours_studied` → `passed`. Store the pass probability for 7 hours in `p7` and for 2 hours in `p2`.

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
hours = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10]).reshape(-1, 1)
passed = np.array([0, 0, 0, 0, 1, 0, 1, 1, 1, 1])
p7 = p2 = None   # TODO


In [ ]:
try:
    check("7 hours -> likely pass", p7 > 0.7)
    check("2 hours -> unlikely", p2 < 0.3)
    check("monotonic", p7 > p2)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
from sklearn.linear_model import LogisticRegression
hours = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10]).reshape(-1, 1)
passed = np.array([0, 0, 0, 0, 1, 0, 1, 1, 1, 1])
m = LogisticRegression().fit(hours, passed)
p7, p2 = m.predict_proba([[7]])[0, 1], m.predict_proba([[2]])[0, 1]

```

</details>

### Exercise 3 · Stretch · Tune the decision threshold

Meridian Bank cares more about catching defaulters (recall). Given true labels and predicted probabilities, search thresholds 0.1, 0.15, …, 0.9 and store the one with the best **F1** in `best_t`, and that F1 in `best_f1`. Then set `beats_default` to whether it beats the F1 at threshold 0.5.

In [ ]:
import numpy as np
from sklearn.metrics import f1_score
rng = np.random.default_rng(6)
y = (rng.random(400) < 0.15).astype(int)
proba = np.clip(0.12 + 0.35 * y + rng.normal(0, 0.12, 400), 0, 1)
best_t = best_f1 = beats_default = None   # TODO


In [ ]:
try:
    check("threshold in range", 0.1 <= best_t <= 0.9)
    check("beats the default 0.5", beats_default is True)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
from sklearn.metrics import f1_score
rng = np.random.default_rng(6)
y = (rng.random(400) < 0.15).astype(int)
proba = np.clip(0.12 + 0.35 * y + rng.normal(0, 0.12, 400), 0, 1)
grid = np.arange(0.1, 0.91, 0.05)
scores = [f1_score(y, proba >= t) for t in grid]
best_t = float(grid[int(np.argmax(scores))])
best_f1 = max(scores)
beats_default = bool(best_f1 > f1_score(y, proba >= 0.5))

```

0.5 is only optimal when classes are balanced and errors cost the same; on rare-event problems the best cut-off is usually lower.

</details>

---
*Back to the course: **Machine Learning End To End → Logistic Regression & Classification**.*